# 🌍 AI Trip Planner with GCP — Full Pipeline Notebook
**End-to-end Colab implementation** mirroring the production project structure.

### Pipeline Stages
| Stage | Description |
|---|---|
| 0 | API Keys from Colab Secrets |
| 1 | Install dependencies |
| 2 | Create project structure |
| 3 | Write `src/utils/` (logger + custom exception) |
| 4 | Write `src/config/config.py` |
| 5 | Write `src/tools/` (Tavily + Serper) |
| 6 | Test search tools individually |
| 7 | Write `src/agents/travel_agent.py` |
| 8 | Write `src/core/planner.py` |
| 9 | Write `app.py` + `setup.py` + `versions.py` |
| 10 | Run full itinerary pipeline & save all outputs |
| 11 | Zip & download everything |

> **LLM:** Groq `llama-3.3-70b-versatile` (primary) · OpenAI `gpt-4o` (fallback)  
> **Search:** Tavily + Google Serper  
> ⚠️ Add `GROQ_API_KEY`, `TAVILY_API_KEY`, `SERPER_API_KEY` to Colab Secrets before running.


## 🔑 Stage 0 — Load API Keys from Colab Secrets

In [1]:
# Stage 0: Load keys from Colab Secrets (userdata)
# In Colab: Runtime → Secrets → add GROQ_API_KEY, TAVILY_API_KEY, SERPER_API_KEY
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")
    print("✅ Keys loaded from Colab Secrets")
except Exception:
    # Fallback: paste keys directly (for local or non-Colab use)
    os.environ["GROQ_API_KEY"]   = "your-groq-api-key-here"
    os.environ["TAVILY_API_KEY"] = "your-tavily-api-key-here"
    os.environ["SERPER_API_KEY"] = "your-serper-api-key-here"
    print("⚠️  Keys set manually — replace with your actual keys")

print("GROQ_API_KEY  :", "set ✅" if os.environ.get("GROQ_API_KEY")   else "MISSING ❌")
print("TAVILY_API_KEY:", "set ✅" if os.environ.get("TAVILY_API_KEY") else "MISSING ❌")
print("SERPER_API_KEY:", "set ✅" if os.environ.get("SERPER_API_KEY") else "MISSING ❌")


✅ Keys loaded from Colab Secrets
GROQ_API_KEY  : set ✅
TAVILY_API_KEY: set ✅
SERPER_API_KEY: set ✅


## ⚙️ Stage 1 — Install Dependencies

In [2]:
# Stage 1: Install all required packages
!pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-openai \
    langchain-tavily \
    langchain-text-splitters \
    python-dotenv \
    pillow

print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ All packages installed


In [3]:
# Verify key package versions (inline)
from importlib.metadata import version, PackageNotFoundError

PKGS = ["langchain","langchain-core","langchain-community",
        "langchain-groq","langchain-tavily","python-dotenv"]

for pkg in PKGS:
    try:
        print(f"  {pkg:35} == {version(pkg)}")
    except PackageNotFoundError:
        print(f"  {pkg:35} NOT INSTALLED")
print("✅ Version check done")


  langchain                           == 1.2.14
  langchain-core                      == 1.2.23
  langchain-community                 == 0.4.1
  langchain-groq                      == 1.1.2
  langchain-tavily                    == 0.2.17
  python-dotenv                       == 1.2.2
✅ Version check done


In [4]:
# Test Tavily API Key
import os
from langchain_tavily import TavilySearch

try:
    tavily = TavilySearch(max_results=1, tavily_api_key=os.environ.get("TAVILY_API_KEY"))
    result = tavily.invoke({"query": "hello world"})
    print("✅ Tavily API Key is VALID")
    print("Sample result:", str(result)[:200])
except Exception as e:
    print("❌ Tavily API Key FAILED:", e)

✅ Tavily API Key is VALID
Sample result: {'query': 'hello world', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.raspberrypi.org/hello-world', 'title': 'Hello World - Raspberry Pi Foundation', 'co


In [5]:
# Test Serper API Key
import os
from langchain_community.utilities import GoogleSerperAPIWrapper

try:
    search = GoogleSerperAPIWrapper(serper_api_key=os.environ.get("SERPER_API_KEY"))
    result = search.run("hello world")
    print("✅ Serper API Key is VALID")
    print("Sample result:", str(result)[:200])
except Exception as e:
    print("❌ Serper API Key FAILED:", e)

✅ Serper API Key is VALID
Sample result: "Hello, World!" program: Computer program. A "Hello, World!" program is usually a simple computer program that displays on the screen a message similar to "Hello, World!". A small piece of code in mos


## 🗂️ Stage 2 — Create Project Structure

In [6]:
# Stage 2: Mirror the production folder structure
import os

dirs = [
    "src/agents",
    "src/config",
    "src/core",
    "src/tools",
    "src/utils",
    "outputs",
    "logs",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# __init__.py files
for pkg in ["src","src/agents","src/config","src/core","src/tools","src/utils"]:
    init = os.path.join(pkg, "__init__.py")
    if not os.path.exists(init):
        open(init, "w").close()

print("✅ Project structure created:")
for d in dirs:
    print(f"   {d}/")


✅ Project structure created:
   src/agents/
   src/config/
   src/core/
   src/tools/
   src/utils/
   outputs/
   logs/


## 🛠️ Stage 3 — Write `src/utils/` (Logger + Custom Exception)

In [7]:
# Stage 3a: Write src/utils/logger.py
logger_code = """
import logging
import os
from datetime import datetime

LOGS_DIR = "logs"
os.makedirs(LOGS_DIR, exist_ok=True)

LOG_FILE = os.path.join(LOGS_DIR, f"log_{datetime.now().strftime('%Y-%m-%d')}.log")

logging.basicConfig(
    filename=LOG_FILE,
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    return logger
"""
with open("src/utils/logger.py", "w") as f:
    f.write(logger_code.strip())
print("✅ src/utils/logger.py written")


✅ src/utils/logger.py written


In [8]:
# Stage 3b: Write src/utils/custom_exception.py
exc_code = """
import sys

class CustomException(Exception):
    def __init__(self, message: str, error_detail: Exception = None):
        self.error_message = self.get_detailed_error_message(message, error_detail)
        super().__init__(self.error_message)

    @staticmethod
    def get_detailed_error_message(message, error_detail):
        _, _, exc_tb = sys.exc_info()
        file_name = exc_tb.tb_frame.f_code.co_filename if exc_tb else "Unknown File"
        line_number = exc_tb.tb_lineno if exc_tb else "Unknown Line"
        return f"{message} | Error: {error_detail} | File: {file_name} | Line: {line_number}"

    def __str__(self):
        return self.error_message
"""
with open("src/utils/custom_exception.py", "w") as f:
    f.write(exc_code.strip())
print("✅ src/utils/custom_exception.py written")


✅ src/utils/custom_exception.py written


In [9]:
# Stage 3c: Verify utils imports
from src.utils.logger import get_logger
from src.utils.custom_exception import CustomException

logger = get_logger("pipeline")
logger.info("Pipeline notebook started")
print("✅ Logger and CustomException imported successfully")


INFO:pipeline:Pipeline notebook started


✅ Logger and CustomException imported successfully


## ⚙️ Stage 4 — Write `src/config/config.py`

In [10]:
# Stage 4: Write src/config/config.py
config_code = """
import os
from src.utils.logger import get_logger

logger = get_logger(__name__)

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
SERPER_API_KEY = os.environ.get("SERPER_API_KEY")

logger.info("Config initialized")
"""
with open("src/config/config.py", "w") as f:
    f.write(config_code.strip())

from src.config.config import GROQ_API_KEY, TAVILY_API_KEY, SERPER_API_KEY
print("✅ src/config/config.py written and imported")
print("   GROQ_API_KEY  :", "set ✅" if GROQ_API_KEY   else "MISSING ❌")
print("   TAVILY_API_KEY:", "set ✅" if TAVILY_API_KEY else "MISSING ❌")
print("   SERPER_API_KEY:", "set ✅" if SERPER_API_KEY else "MISSING ❌")

INFO:src.config.config:Config initialized


✅ src/config/config.py written and imported
   GROQ_API_KEY  : set ✅
   TAVILY_API_KEY: set ✅
   SERPER_API_KEY: set ✅


## 🔍 Stage 5 — Write `src/tools/` (Tavily + Serper Search Tools)

In [11]:
# Stage 5a: Write src/tools/tavily_tool.py
import textwrap

tavily_code = '''
from langchain_tavily import TavilySearch
from src.config.config import TAVILY_API_KEY
from src.utils.logger import get_logger

logger = get_logger(__name__)

def tavily_search_tool(query: str) -> str:
    """
    Search the web using Tavily to get up-to-date travel information,
    attractions, activities, tips, and local insights for a given query.
    """
    tavily_search = TavilySearch(
        max_results=5,
        topic="general",
        tavily_api_key=TAVILY_API_KEY
    )
    return tavily_search.invoke({"query": query})

logger.info("Tavily tool ready")
'''

with open("src/tools/tavily_tool.py", "w") as f:
    f.write(textwrap.dedent(tavily_code).strip())

print("✅ src/tools/tavily_tool.py written")

✅ src/tools/tavily_tool.py written


In [12]:
# Stage 5b: Write src/tools/serper_tool.py
import textwrap

serper_code = '''
from langchain_community.utilities import GoogleSerperAPIWrapper
from src.config.config import SERPER_API_KEY
from src.utils.logger import get_logger

logger = get_logger(__name__)

def google_serper_search_tool(query: str) -> str:
    """
    Search Google via Serper API to fetch recent and reliable
    real-world travel information for the given query.
    """
    search = GoogleSerperAPIWrapper(serper_api_key=SERPER_API_KEY)
    return search.run(query)

logger.info("Serper tool ready")
'''

with open("src/tools/serper_tool.py", "w") as f:
    f.write(textwrap.dedent(serper_code).strip())

print("✅ src/tools/serper_tool.py written")

✅ src/tools/serper_tool.py written


## 🧪 Stage 6 — Test Search Tools Individually

In [13]:
# Stage 6a: Test Tavily search tool
from src.tools.tavily_tool import tavily_search_tool

print("🔍 Testing Tavily search...")
tavily_result = tavily_search_tool("best places to visit in Kasol travel guide")
print("✅ Tavily result (first 500 chars):")
print(str(tavily_result)[:500])

# Save output
with open("outputs/stage6a_tavily_test.txt", "w") as f:
    f.write(str(tavily_result))
print("💾 Saved → outputs/stage6a_tavily_test.txt")

INFO:src.tools.tavily_tool:Tavily tool ready


🔍 Testing Tavily search...
✅ Tavily result (first 500 chars):
{'query': 'best places to visit in Kasol travel guide', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.thomascook.in/blog/india-holidays/places-to-visit-in-kasol/', 'title': 'Top 10 Places to Visit in Kasol for Backpackers & Trekkers', 'content': '###### Blog by theme. ###### Blog By Theme. # Top 10 Places to Visit in Kasol for Nature Lovers and Backpackers. ### Table of contents. Tosh is the starting point of the trek. While you may not want to trek 
💾 Saved → outputs/stage6a_tavily_test.txt


In [14]:
# Stage 6b: Test Google Serper search tool
from src.tools.serper_tool import google_serper_search_tool

print("🔍 Testing Google Serper search...")
serper_result = google_serper_search_tool("best places to visit in Kasol travel guide")
print("✅ Serper result (first 500 chars):")
print(str(serper_result)[:500])

# Save output
with open("outputs/stage6b_serper_test.txt", "w") as f:
    f.write(str(serper_result))
print("💾 Saved → outputs/stage6b_serper_test.txt")

INFO:src.tools.serper_tool:Serper tool ready


🔍 Testing Google Serper search...
✅ Serper result (first 500 chars):
Top Attractions in Kasol ; 1. Sar Pass Trek · 4.9. (14). Hiking Trails ; 2. Parvati Valley. Mountains ; 3. Ayur Spa · 4.8. (5). Spas ; 4. Psychedelic Painting Shop. Further out, at a distance of 75 km, there's Kullu, a town known for its beautiful apple orchards, ancient temples and the Great Himalayan National Park. Kasol travel recommendations for a 2-day trip. Best cafes in Kasol for views and food. Sightseeing places in Kasol. Hidden gems for solo ... Kasol Travel Guide: Where To Go, Eat, Sh
💾 Saved → outputs/stage6b_serper_test.txt


## 🤖 Stage 7 — Write `src/agents/travel_agent.py`
**Primary LLM:** Groq `llama-3.3-70b-versatile`  
**Fallback LLM:** OpenAI `gpt-4o` (set `USE_OPENAI = True` below if needed)


In [15]:
# Stage 7: Write src/agents/travel_agent.py
agent_code = """
import os
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from src.tools.tavily_tool import tavily_search_tool
from src.tools.serper_tool import google_serper_search_tool
from src.utils.logger import get_logger

logger = get_logger(__name__)

# ── LLM selection ──────────────────────────────────────────────────────────
# Primary: Groq (free, fast)  |  Fallback: OpenAI gpt-4o
USE_OPENAI = False  # Set True to switch to OpenAI gpt-4o

if USE_OPENAI:
    model = init_chat_model(
        model="openai:gpt-4o",
        temperature=0.3
    )
    logger.info("Using OpenAI gpt-4o")
else:
    model = init_chat_model(
        model="groq:llama-3.3-70b-versatile",
        temperature=0.3
    )
    logger.info("Using Groq llama-3.3-70b-versatile")

SYSTEM_PROMPT = '''
You are an expert AI travel planner.

Rules:
- Always use web search tools for real-world accuracy
- Create a detailed day-wise itinerary
- Include food suggestions, local tips, and travel advice

User Inputs:
City, Number of days, Interests, Travel style, Pace
'''.strip()

agent = create_react_agent(
    model=model,
    tools=[tavily_search_tool, google_serper_search_tool],
    prompt=SYSTEM_PROMPT
)

logger.info("Travel agent created")
"""
with open("src/agents/travel_agent.py", "w") as f:
    f.write(agent_code.strip())
print("✅ src/agents/travel_agent.py written")

✅ src/agents/travel_agent.py written


In [16]:
# Stage 7b: Initialize the agent (takes ~10s)
from src.agents.travel_agent import agent, model
print("✅ Travel agent initialized")
print("   Model:", model)


INFO:src.agents.travel_agent:Using Groq llama-3.3-70b-versatile
INFO:src.agents.travel_agent:Travel agent created


✅ Travel agent initialized
   Model: profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x7c6fb1e67650> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c6fb1075670> model_name='llama-3.3-70b-versatile' temperature=0.3 model_kwargs={} groq_api_key=SecretStr('**********')


## 🗺️ Stage 8 — Write `src/core/planner.py`

In [17]:
# Stage 8: Write src/core/planner.py
planner_code = """
from langchain_core.messages import HumanMessage, AIMessage
from src.agents.travel_agent import agent
from src.utils.logger import get_logger
from src.utils.custom_exception import CustomException

logger = get_logger(__name__)

class TravelPlanner:
    def __init__(self):
        self.messages = []
        logger.info("TravelPlanner initialized")

    def create_itinerary(
        self,
        city: str,
        days: int,
        interests: list,
        style: str,
        pace: str,
        month: str = None
    ) -> str:
        try:
            user_prompt = f\"\"\"
            Plan a {days}-day trip to {city}

            Interests: {', '.join(interests)}
            Travel Style: {style}
            Pace: {pace}
            Month: {month or 'Any'}

            Provide a detailed day-wise itinerary with:
            - Morning, afternoon and evening activities for each day
            - Food and restaurant recommendations
            - Local tips and travel advice
            - Estimated costs where possible
            \"\"\".strip()

            self.messages.append(HumanMessage(content=user_prompt))
            response = agent.invoke({"messages": self.messages})

            # Safe extraction: find last AIMessage with content
            final_answer = None
            for message in reversed(response["messages"]):
                if isinstance(message, AIMessage) and message.content:
                    final_answer = message.content
                    break

            if not final_answer:
                final_answer = response["messages"][-1].content

            self.messages.append(AIMessage(content=final_answer))
            logger.info(f"Itinerary generated for {city}")
            return final_answer

        except Exception as e:
            logger.error(f"Planner error: {e}")
            raise CustomException("Failed to generate itinerary", e)
"""
with open("src/core/planner.py", "w") as f:
    f.write(planner_code.strip())

from src.core.planner import TravelPlanner
print("✅ src/core/planner.py written and TravelPlanner imported")


✅ src/core/planner.py written and TravelPlanner imported


## 🖥️ Stage 9 — Write `app.py`, `setup.py`, `versions.py`

In [18]:
# Stage 9a: Write app.py (Streamlit frontend)
app_code = """
import warnings
warnings.filterwarnings("ignore")

import os
import streamlit as st
from src.core.planner import TravelPlanner
from src.utils.logger import get_logger

logger = get_logger(__name__)

st.set_page_config(page_title="AI Travel Planner", layout="wide")
st.title("🌍 AI Travel Itinerary Planner")

with st.form("planner_form"):
    city      = st.text_input("📍 City")
    days      = st.slider("🗓️ Number of days", 1, 10, 3)
    interests = st.text_input("🎯 Interests (comma-separated)")
    style     = st.selectbox("💰 Travel Style", ["Budget", "Mid-range", "Luxury"])
    pace      = st.selectbox("🚶 Pace", ["Relaxed", "Balanced", "Packed"])
    month     = st.selectbox("📅 Month (optional)", ["Any","Jan","Feb","Mar","Apr",
                              "May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
    submitted = st.form_submit_button("✨ Generate Itinerary")

if submitted:
    if city and interests:
        with st.spinner("🧠 Planning your trip..."):
            planner   = TravelPlanner()
            itinerary = planner.create_itinerary(
                city=city,
                days=days,
                interests=[i.strip() for i in interests.split(",")],
                style=style,
                pace=pace,
                month=None if month == "Any" else month
            )
        st.subheader("📄 Your Travel Plan")
        st.markdown(itinerary)
        logger.info("Response generated")
    else:
        st.warning("Please enter city and interests")
"""
with open("app.py", "w") as f:
    f.write(app_code.strip())
print("✅ app.py written")


✅ app.py written


In [19]:
# Stage 9b: Write setup.py
setup_code = """
from setuptools import setup, find_packages

with open("requirements.txt") as f:
    requirements = [l for l in f.read().splitlines()
                    if l and not l.startswith("#") and not l.startswith("uv")]

setup(
    name="AI Trip Planner",
    version="0.1",
    author="divesh",
    packages=find_packages(),
    install_requires=requirements,
)
"""
with open("setup.py", "w") as f:
    f.write(setup_code.strip())
print("✅ setup.py written")


✅ setup.py written


In [20]:
# Stage 9c: Write versions.py and run it
versions_code = (
    "from importlib.metadata import version, PackageNotFoundError\n"
    "\n"
    "PACKAGES = [\n"
    "    'langchain', 'langchain-core', 'langchain-community',\n"
    "    'langchain-groq', 'langchain-tavily',\n"
    "    'langchain-text-splitters', 'python-dotenv',\n"
    "]\n"
    "\n"
    "def get_package_version(pkg_name):\n"
    "    try:\n"
    "        return version(pkg_name)\n"
    "    except PackageNotFoundError:\n"
    "        return 'NOT INSTALLED'\n"
    "\n"
    "def print_versions():\n"
    "    versions = {pkg: get_package_version(pkg) for pkg in PACKAGES}\n"
    "    return versions\n"
    "\n"
    "if __name__ == '__main__':\n"
    "    v = print_versions()\n"
    "    print('\\n\U0001f4e6 Installed Package Versions:\\n')\n"
    "    for k, val in v.items():\n"
    "        print(f'  {k:35} == {val}')\n"
    "    print('')\n"
)
with open("versions.py", "w") as f:
    f.write(versions_code)

# Run it
import importlib.util, sys
spec = importlib.util.spec_from_file_location("versions", "versions.py")
vm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vm)
v = vm.print_versions()
print("\n\U0001f4e6 Package Versions:")
for k, val in v.items():
    print(f"  {k:35} == {val}")
print("\n\u2705 versions.py written and executed")


📦 Package Versions:
  langchain                           == 1.2.14
  langchain-core                      == 1.2.23
  langchain-community                 == 0.4.1
  langchain-groq                      == 1.1.2
  langchain-tavily                    == 0.2.17
  langchain-text-splitters            == 1.1.1
  python-dotenv                       == 1.2.2

✅ versions.py written and executed


## 🚀 Stage 10 — Run Full Itinerary Pipeline & Save All Outputs
Three sample queries are run and every output is saved to `outputs/`.


In [21]:
# Stage 10a: Query 1 — Kasol, 3 days, Budget, Relaxed
from src.core.planner import TravelPlanner
import json

planner = TravelPlanner()

print("🌄 Query 1: Kasol — 3 days | Budget | Relaxed | Interests: Trekking, Nature, Cafes")
itinerary_1 = planner.create_itinerary(
    city="Kasol, Himachal Pradesh",
    days=3,
    interests=["Trekking", "Nature", "Cafes", "Backpacking"],
    style="Budget",
    pace="Relaxed",
    month="May"
)
print("\n" + "="*60)
print(itinerary_1)
print("="*60)

with open("outputs/stage10a_query1_kasol.txt", "w") as f:
    f.write(itinerary_1)
print("\n💾 Saved → outputs/stage10a_query1_kasol.txt")


INFO:src.core.planner:TravelPlanner initialized


🌄 Query 1: Kasol — 3 days | Budget | Relaxed | Interests: Trekking, Nature, Cafes


INFO:src.core.planner:Itinerary generated for Kasol, Himachal Pradesh



Here's a detailed day-wise itinerary for a 3-day trip to Kasol, Himachal Pradesh:

Day 1:
- Morning: Arrive in Kasol and check into a budget-friendly hotel or hostel. Explore the Kasol market and try some local food at a café.
- Afternoon: Take a short hike to Chalal village, which is a scenic and peaceful destination. Visit the Gurudwara Manikaran Sahib in the afternoon.
- Evening: Visit some nice cafes in Kasol, such as the Evergreen Café or the Moonlight Café, and enjoy the scenic views of the Parvati Valley.

Day 2:
- Morning: Start early for the Kheerganga trek, which is a popular trekking route in Kasol. The trek takes around 4-5 hours to complete and offers stunning views of the valley.
- Afternoon: Reach the top of the trek and enjoy the scenic views. Have lunch at a local café or restaurant.
- Evening: Return to Kasol and visit the Kasol market for some shopping and exploration.

Day 3:
- Morning: Visit the village of Tosh, which is known for its scenic views and trekking rou

In [22]:
# Stage 10b: Query 2 — Goa, 5 days, Mid-range, Balanced
planner2 = TravelPlanner()

print("🏖️ Query 2: Goa — 5 days | Mid-range | Balanced | Interests: Beaches, Food, Nightlife")
itinerary_2 = planner2.create_itinerary(
    city="Goa",
    days=5,
    interests=["Beaches", "Food", "Nightlife", "Water Sports"],
    style="Mid-range",
    pace="Balanced",
    month="Dec"
)
print("\n" + "="*60)
print(itinerary_2)
print("="*60)

with open("outputs/stage10b_query2_goa.txt", "w") as f:
    f.write(itinerary_2)
print("\n💾 Saved → outputs/stage10b_query2_goa.txt")


INFO:src.core.planner:TravelPlanner initialized


🏖️ Query 2: Goa — 5 days | Mid-range | Balanced | Interests: Beaches, Food, Nightlife


INFO:src.core.planner:Itinerary generated for Goa



Here's a detailed day-wise itinerary for a 5-day trip to Goa:

Day 1: Arrival and Relaxation

* Morning: Arrive at Dabolim Airport and take a taxi or bus to your hotel in North Goa.
* Afternoon: Check-in to your hotel and relax on the beach or by the pool.
* Evening: Head to Baga Beach for a sunset view and try some street food at the local stalls.
* Food: Try some traditional Goan cuisine like fish curry and rice at a local restaurant.
* Local tip: Hire a scooter to explore the area and visit the nearby Anjuna Market.

Day 2: Beach Hopping

* Morning: Start the day with a visit to Calangute Beach, one of the most popular beaches in North Goa.
* Afternoon: Head to Vagator Beach, known for its scenic views and water sports.
* Evening: Enjoy a sunset view at Morjim Beach and try some seafood at a local restaurant.
* Food: Try some fresh seafood at a beachside restaurant.
* Local tip: Be sure to try some water sports like parasailing or jet-skiing at Vagator Beach.

Day 3: Water Sports a

In [23]:
# Stage 10c: Query 3 — Rajasthan, 7 days, Luxury, Packed
planner3 = TravelPlanner()

print("🏰 Query 3: Rajasthan — 7 days | Luxury | Packed | Interests: History, Culture, Cuisine")
itinerary_3 = planner3.create_itinerary(
    city="Rajasthan (Jaipur, Jodhpur, Udaipur)",
    days=7,
    interests=["History", "Culture", "Cuisine", "Architecture", "Desert Safari"],
    style="Luxury",
    pace="Packed",
    month="Oct"
)
print("\n" + "="*60)
print(itinerary_3)
print("="*60)

with open("outputs/stage10c_query3_rajasthan.txt", "w") as f:
    f.write(itinerary_3)
print("\n💾 Saved → outputs/stage10c_query3_rajasthan.txt")


INFO:src.core.planner:TravelPlanner initialized


🏰 Query 3: Rajasthan — 7 days | Luxury | Packed | Interests: History, Culture, Cuisine


INFO:src.core.planner:Itinerary generated for Rajasthan (Jaipur, Jodhpur, Udaipur)



Here's a 7-day itinerary for a luxury trip to Rajasthan, covering Jaipur, Jodhpur, and Udaipur:

Day 1: Arrival in Jaipur
- Morning: Arrive at Jaipur airport and check-in to a luxury hotel like ITC Rajputana or The Leela Palace Jaipur.
- Afternoon: Visit the City Palace and Jantar Mantar.
- Evening: Enjoy a traditional Rajasthani dinner at a local restaurant like Chokhi Dhani.

Day 2: Jaipur
- Morning: Visit the Amber Fort and take an elephant ride.
- Afternoon: Explore the Johri Bazaar and buy some local handicrafts.
- Evening: Enjoy a cultural performance at a local theater.

Day 3: Jaipur to Jodhpur
- Morning: Drive to Jodhpur (around 5 hours) and check-in to a luxury hotel like Umaid Bhawan Palace or Taj Hari Mahal.
- Afternoon: Visit the Mehrangarh Fort and explore the Blue City.
- Evening: Enjoy a sunset view of the fort and the city.

Day 4: Jodhpur
- Morning: Visit the Jaswant Thada and the Umaid Bhawan Palace Museum.
- Afternoon: Take a desert safari and enjoy the dunes.
- Ev

In [24]:
# Stage 10d: Save full pipeline summary
import json
from datetime import datetime

summary = {
    "pipeline_run": datetime.now().isoformat(),
    "queries": [
        {"query": "Kasol 3-day Budget trip", "output": "outputs/stage10a_query1_kasol.txt"},
        {"query": "Goa 5-day Mid-range trip", "output": "outputs/stage10b_query2_goa.txt"},
        {"query": "Rajasthan 7-day Luxury trip", "output": "outputs/stage10c_query3_rajasthan.txt"},
    ],
    "tools_tested": [
        {"tool": "Tavily", "output": "outputs/stage6a_tavily_test.txt"},
        {"tool": "Serper", "output": "outputs/stage6b_serper_test.txt"},
    ]
}

with open("outputs/stage10d_pipeline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Full pipeline complete!")
print("💾 Saved → outputs/stage10d_pipeline_summary.json")
print("\n📂 All outputs:")
import os
for fn in sorted(os.listdir("outputs")):
    fpath = os.path.join("outputs", fn)
    size = os.path.getsize(fpath)
    print(f"   {fn:50} {size:>8} bytes")


✅ Full pipeline complete!
💾 Saved → outputs/stage10d_pipeline_summary.json

📂 All outputs:
   stage10a_query1_kasol.txt                              2609 bytes
   stage10b_query2_goa.txt                                2911 bytes
   stage10c_query3_rajasthan.txt                          1937 bytes
   stage10d_pipeline_summary.json                          595 bytes
   stage6a_tavily_test.txt                                4627 bytes
   stage6b_serper_test.txt                                1533 bytes


## 📦 Stage 11 — Zip & Download All Outputs

In [25]:
# Stage 11: Zip entire project + outputs and download
import zipfile, os
from google.colab import files

zip_path = "AI_Trip_Planner_GCP_Outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Source files
    for fname in ["app.py", "setup.py", "versions.py", "requirements.txt"]:
        if os.path.exists(fname):
            zf.write(fname)
    # src/ package
    for root, dirs, fnames in os.walk("src"):
        dirs[:] = [d for d in dirs if d != "__pycache__"]
        for fn in fnames:
            if not fn.endswith(".pyc"):
                zf.write(os.path.join(root, fn))
    # outputs/
    for root, dirs, fnames in os.walk("outputs"):
        for fn in fnames:
            zf.write(os.path.join(root, fn))
    # logs/
    for root, dirs, fnames in os.walk("logs"):
        for fn in fnames:
            zf.write(os.path.join(root, fn))

print(f"✅ Zipped: {zip_path}")
print("\n📂 Contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in sorted(zf.namelist()):
        print(f"   {name}")

files.download(zip_path)
print("\n⬇️  Download started!")


✅ Zipped: AI_Trip_Planner_GCP_Outputs.zip

📂 Contents:
   app.py
   outputs/stage10a_query1_kasol.txt
   outputs/stage10b_query2_goa.txt
   outputs/stage10c_query3_rajasthan.txt
   outputs/stage10d_pipeline_summary.json
   outputs/stage6a_tavily_test.txt
   outputs/stage6b_serper_test.txt
   setup.py
   src/__init__.py
   src/agents/__init__.py
   src/agents/travel_agent.py
   src/config/__init__.py
   src/config/config.py
   src/core/__init__.py
   src/core/planner.py
   src/tools/__init__.py
   src/tools/serper_tool.py
   src/tools/tavily_tool.py
   src/utils/__init__.py
   src/utils/custom_exception.py
   src/utils/logger.py
   versions.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️  Download started!
